# ANRF AISEHack 2.0 — Polymer Property Prediction v8
**v4/v6 public LB: 0.896 | OOF: 0.9117  (Tg: 0.9082, Egc: 0.9151)**

**v8 adds a Graph Neural Network (AttentiveFP):**
- Operates directly on the molecular graph (atoms + bonds) — no information loss from fingerprint hashing
- Error correlation with XGB measured at **0.765** on a 600-sample test — meaningfully more diverse than ExtraTrees (0.82, which got 0 blend weight) or LightGBM (which converges toward XGB after seed averaging)
- On that same 600-sample test, blending GNN + XGB gave **+0.0198 R²** over pure XGB
- ExtraTrees and CatBoost dropped — both got 0.000 blend weight in earlier versions
- Final ensemble: **LGBM + XGB + GNN**, 3-way weights optimised on seed-averaged OOF
- GNN runtime is small (~25 min for 5-fold × 1 seed on both targets combined) — runs 3 seeds for stability with room to spare


In [ ]:
!pip install rdkit -q
!pip install torch torch_geometric -q

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import glob

from rdkit import Chem
from rdkit.Chem import Descriptors, MACCSkeys, rdFingerprintGenerator, RDKFingerprint

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

import lightgbm as lgb
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

import torch
import torch.nn as nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn.models import AttentiveFP

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Torch device: {DEVICE}')


In [ ]:
train_path = glob.glob('/kaggle/input/**/train.csv', recursive=True)[0]
test_path  = glob.glob('/kaggle/input/**/test.csv',  recursive=True)[0]

train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

train_tg  = train[train['target_type'] == 'tg'].reset_index(drop=True)
train_egc = train[train['target_type'] == 'egc'].reset_index(drop=True)
test_tg   = test[test['target_type'] == 'tg'].reset_index(drop=True)
test_egc  = test[test['target_type'] == 'egc'].reset_index(drop=True)

y_tg  = train_tg['target'].values
y_egc = train_egc['target'].values

print(f'Tg  train: {len(train_tg):,}  test: {len(test_tg):,}')
print(f'Egc train: {len(train_egc):,}  test: {len(test_egc):,}')
print(f'Tg  range: [{y_tg.min():.1f}, {y_tg.max():.1f}]')
print(f'Egc range: [{y_egc.min():.4f}, {y_egc.max():.4f}]')

## Feature Engineering

**v2 adds two new fingerprint types on top of v1:**
- **ECFP6** (Morgan radius=3, 2048 bits): larger circular neighbourhoods than ECFP4 — captures longer-range substructures important for Tg
- **RDKit topological fingerprints** (2048 bits): path-based rather than circular; complementary information

Total raw features: ~4,500 (up from ~2,400). Variance filtering still applied.

In [ ]:
DESC_NAMES  = [n for n, _ in Descriptors.descList]
MORGAN_ECFP4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
MORGAN_ECFP6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)
RDKIT_FPGEN  = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048)


def featurize(smiles_list):
    rdkit_rows, ecfp4_rows, ecfp6_rows, rdk_rows, maccs_rows = [], [], [], [], []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rdkit_rows.append([np.nan] * len(DESC_NAMES))
            ecfp4_rows.append(np.zeros(2048, dtype=np.uint8))
            ecfp6_rows.append(np.zeros(2048, dtype=np.uint8))
            rdk_rows.append(np.zeros(2048, dtype=np.uint8))
            maccs_rows.append(np.zeros(167, dtype=np.uint8))
        else:
            vals = Descriptors.CalcMolDescriptors(mol)
            rdkit_rows.append(list(vals.values()))
            ecfp4_rows.append(MORGAN_ECFP4.GetFingerprintAsNumPy(mol))
            ecfp6_rows.append(MORGAN_ECFP6.GetFingerprintAsNumPy(mol))
            rdk_rows.append(RDKIT_FPGEN.GetFingerprintAsNumPy(mol))
            fp_mac = MACCSkeys.GenMACCSKeys(mol)
            maccs_rows.append(np.array(fp_mac, dtype=np.uint8))

    return pd.concat([
        pd.DataFrame(rdkit_rows,  columns=DESC_NAMES),
        pd.DataFrame(ecfp4_rows,  columns=[f'ecfp4_{i}'  for i in range(2048)]),
        pd.DataFrame(ecfp6_rows,  columns=[f'ecfp6_{i}'  for i in range(2048)]),
        pd.DataFrame(rdk_rows,    columns=[f'rdkfp_{i}'  for i in range(2048)]),
        pd.DataFrame(maccs_rows,  columns=[f'maccs_{i}'  for i in range(167)]),
    ], axis=1)


def build_preprocessor(X_raw):
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.dropna(axis=1, thresh=int(0.2 * len(X)))
    X = X.loc[:, X.var() > 0]
    good_cols = X.columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    X_imp    = imputer.fit_transform(X)
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_imp)
    return X_scaled, (imputer, scaler, good_cols)


def apply_preprocessor(X_raw, preprocessor):
    imputer, scaler, good_cols = preprocessor
    X = X_raw.replace([np.inf, -np.inf], np.nan).astype(float)
    X = X.reindex(columns=good_cols, fill_value=np.nan)
    return scaler.transform(imputer.transform(X))


print('Feature functions defined.')

In [ ]:
print('Featurizing Tg train  ...', flush=True)
X_tg_raw       = featurize(train_tg['smiles'].tolist())
print('Featurizing Egc train ...', flush=True)
X_egc_raw      = featurize(train_egc['smiles'].tolist())
print('Featurizing Tg test   ...', flush=True)
X_tg_test_raw  = featurize(test_tg['smiles'].tolist())
print('Featurizing Egc test  ...', flush=True)
X_egc_test_raw = featurize(test_egc['smiles'].tolist())

print(f'\nRaw feature shape: {X_tg_raw.shape}')
print('Preprocessing ...')

X_tg,  tg_prep  = build_preprocessor(X_tg_raw)
X_egc, egc_prep = build_preprocessor(X_egc_raw)
X_tg_test  = apply_preprocessor(X_tg_test_raw,  tg_prep)
X_egc_test = apply_preprocessor(X_egc_test_raw, egc_prep)

print(f'Tg  features after cleaning: {X_tg.shape[1]:,}')
print(f'Egc features after cleaning: {X_egc.shape[1]:,}')

## Graph Neural Network (AttentiveFP)

AttentiveFP (Xiong et al., 2020) is a graph attention network designed specifically for molecular property prediction. Unlike fingerprints, it operates on the actual atom-bond graph, so structural information isn't lost to feature hashing.

**Atom features (18-dim):** element one-hot (13 categories incl. `*` and 'Other'), degree, formal charge, aromaticity, H-count, hybridization.

**Bond features (6-dim):** bond order one-hot (single/double/triple/aromatic), conjugation, ring membership.

**Architecture:** 2 attention layers, 2 readout timesteps, 64 hidden channels — kept small since the dataset is modest in size (overly large GNNs overfit fast here).

In [ ]:
ATOM_LIST = ['C','N','O','S','F','Si','P','Cl','Br','I','B','*','Other']


def one_hot(val, choices):
    vec = [0] * len(choices)
    idx = choices.index(val) if val in choices else len(choices) - 1
    vec[idx] = 1
    return vec


def atom_features(atom):
    feats = one_hot(atom.GetSymbol(), ATOM_LIST)
    feats += [
        atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(), int(atom.GetHybridization()),
    ]
    return feats


def bond_features(bond):
    bt = bond.GetBondType()
    return [
        int(bt == Chem.rdchem.BondType.SINGLE),
        int(bt == Chem.rdchem.BondType.DOUBLE),
        int(bt == Chem.rdchem.BondType.TRIPLE),
        int(bt == Chem.rdchem.BondType.AROMATIC),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
    ]


def smiles_to_graph(smiles, y=None):
    """Convert a SMILES string to a PyG Data object. Returns None if invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)

    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bf, bf]

    if len(edge_index) == 0:          # single-atom edge case
        edge_index = [[0, 0]]
        edge_attr  = [[0] * 6]

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    return data


NODE_DIM = len(ATOM_LIST) + 5
EDGE_DIM = 6
print(f'NODE_DIM={NODE_DIM}  EDGE_DIM={EDGE_DIM}')


In [ ]:
from sklearn.preprocessing import StandardScaler

# Target scaling helps GNN training stability (similar scale gradients across targets)
y_tg_scaler  = StandardScaler()
y_egc_scaler = StandardScaler()
y_tg_scaled  = y_tg_scaler.fit_transform(y_tg.reshape(-1, 1)).ravel()
y_egc_scaled = y_egc_scaler.fit_transform(y_egc.reshape(-1, 1)).ravel()

print('Building Tg graphs ...')
graphs_tg = [smiles_to_graph(s, y) for s, y in zip(train_tg['smiles'], y_tg_scaled)]
graphs_tg_test = [smiles_to_graph(s) for s in test_tg['smiles']]

print('Building Egc graphs ...')
graphs_egc = [smiles_to_graph(s, y) for s, y in zip(train_egc['smiles'], y_egc_scaled)]
graphs_egc_test = [smiles_to_graph(s) for s in test_egc['smiles']]

print(f'Tg  graphs: {len(graphs_tg)} train, {len(graphs_tg_test)} test')
print(f'Egc graphs: {len(graphs_egc)} train, {len(graphs_egc_test)} test')


In [ ]:
GNN_SEEDS = [42, 7, 123]   # 3 seeds — GNN is fast, room for more if time allows


def train_gnn_ensemble(graphs, y_raw, graphs_test, scaler, seed, n_splits=5,
                       hidden=64, layers=2, timesteps=2, epochs=60, patience=12, lr=1e-3):
    """
    n_splits-fold AttentiveFP ensemble for one seed.
    Returns (oof_predictions_in_original_units, test_predictions_in_original_units).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    idx = np.arange(len(graphs))
    oof_scaled  = np.zeros(len(graphs))
    test_scaled = np.zeros(len(graphs_test))

    test_loader = DataLoader(graphs_test, batch_size=256, shuffle=False)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(idx), 1):
        train_loader = DataLoader([graphs[i] for i in tr_idx], batch_size=64, shuffle=True)
        val_loader   = DataLoader([graphs[i] for i in val_idx], batch_size=128, shuffle=False)

        model = AttentiveFP(
            in_channels=NODE_DIM, hidden_channels=hidden, out_channels=1,
            edge_dim=EDGE_DIM, num_layers=layers, num_timesteps=timesteps, dropout=0.1
        ).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        loss_fn = nn.MSELoss()

        best_val_loss, bad_epochs, best_state = float('inf'), 0, None

        for epoch in range(epochs):
            model.train()
            for batch in train_loader:
                batch = batch.to(DEVICE)
                opt.zero_grad()
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                loss = loss_fn(out.squeeze(-1), batch.y)
                loss.backward()
                opt.step()

            model.eval()
            val_losses = []
            with torch.no_grad():
                for batch in val_loader:
                    batch = batch.to(DEVICE)
                    out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                    val_losses.append(loss_fn(out.squeeze(-1), batch.y).item())
            val_loss = np.mean(val_losses)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

        model.load_state_dict(best_state)
        model.eval()

        val_preds = []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                val_preds.append(out.squeeze(-1).cpu().numpy())
        oof_scaled[val_idx] = np.concatenate(val_preds)

        test_preds = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(DEVICE)
                out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
                test_preds.append(out.squeeze(-1).cpu().numpy())
        test_scaled += np.concatenate(test_preds) / n_splits

        r2_fold = r2_score(y_raw[val_idx], scaler.inverse_transform(
            oof_scaled[val_idx].reshape(-1, 1)).ravel())
        print(f'    fold {fold} | epochs={epoch+1:2d}  GNN R²={r2_fold:.4f}')

    oof  = scaler.inverse_transform(oof_scaled.reshape(-1, 1)).ravel()
    test = scaler.inverse_transform(test_scaled.reshape(-1, 1)).ravel()
    print(f'  OOF GNN R²: {r2_score(y_raw, oof):.4f}')
    return oof, test


print('GNN training function defined.')


In [ ]:
print('=' * 60)
print('  GNN — Tg')
print('=' * 60)

gnn_oof_tg_all, gnn_test_tg_all = [], []
for seed in GNN_SEEDS:
    print(f'\n  seed={seed}')
    oof_g, test_g = train_gnn_ensemble(graphs_tg, y_tg, graphs_tg_test, y_tg_scaler, seed=seed)
    gnn_oof_tg_all.append(oof_g)
    gnn_test_tg_all.append(test_g)

oof_g_tg  = np.mean(gnn_oof_tg_all,  axis=0)
test_g_tg = np.mean(gnn_test_tg_all, axis=0)
print(f'\nSeed-averaged GNN OOF R² (Tg): {r2_score(y_tg, oof_g_tg):.4f}')


In [ ]:
print('=' * 60)
print('  GNN — Egc')
print('=' * 60)

gnn_oof_egc_all, gnn_test_egc_all = [], []
for seed in GNN_SEEDS:
    print(f'\n  seed={seed}')
    oof_g, test_g = train_gnn_ensemble(graphs_egc, y_egc, graphs_egc_test, y_egc_scaler, seed=seed)
    gnn_oof_egc_all.append(oof_g)
    gnn_test_egc_all.append(test_g)

oof_g_egc  = np.mean(gnn_oof_egc_all,  axis=0)
test_g_egc = np.mean(gnn_test_egc_all, axis=0)
print(f'\nSeed-averaged GNN OOF R² (Egc): {r2_score(y_egc, oof_g_egc):.4f}')


## Tabular Ensemble: LGBM + XGB

ExtraTrees and CatBoost were dropped after v6/v2 analysis — both received 0.000 blend weight. 10-fold CV × 5 seeds, same as v4/v6.

In [ ]:
SEEDS = [42, 7, 123, 17, 99]

def lgbm_params(target_type):
    p = dict(
        objective='regression', metric='rmse',
        n_estimators=4000, learning_rate=0.01,
        num_leaves=127, max_depth=-1,
        min_child_samples=15,
        subsample=0.8, subsample_freq=1,
        colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, verbose=-1,
    )
    if target_type == 'egc':
        p['num_leaves'] = 63
        p['min_child_samples'] = 20
    return p


def xgb_params(target_type):
    p = dict(
        objective='reg:squarederror',
        n_estimators=4000, learning_rate=0.01,
        max_depth=6, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.4,
        reg_alpha=0.05, reg_lambda=1.0,
        n_jobs=-1, tree_method='hist',
        early_stopping_rounds=200,
    )
    if target_type == 'egc':
        p['max_depth'] = 5
    return p


print('Model config ready — LGBM + XGB, 5 seeds, 10 folds.') — LGBM + XGB + ExtraTrees, 5 seeds, 10 folds.')

In [ ]:
def train_ensemble(X_train, y_train, X_test, target_type, seed, n_splits=10):
    """
    10-fold CV ensemble of LGBM + XGB for one seed.
    Returns (oof_l, oof_x), (test_l, test_x).
    """
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train)
    X_test  = np.asarray(X_test)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_lgbm  = np.zeros(len(X_train))
    oof_xgb   = np.zeros(len(X_train))
    test_lgbm = np.zeros(len(X_test))
    test_xgb  = np.zeros(len(X_test))

    lp = lgbm_params(target_type); lp['random_state'] = seed
    xp = xgb_params(target_type);  xp['random_state'] = seed

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        m_lgbm = lgb.LGBMRegressor(**lp)
        m_lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                   callbacks=[lgb.early_stopping(200, verbose=False),
                              lgb.log_evaluation(period=0)])
        oof_lgbm[val_idx] = m_lgbm.predict(X_val)
        test_lgbm        += m_lgbm.predict(X_test) / n_splits

        m_xgb = xgb.XGBRegressor(**xp)
        m_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict(X_val)
        test_xgb        += m_xgb.predict(X_test) / n_splits

        r2_l = r2_score(y_val, oof_lgbm[val_idx])
        r2_x = r2_score(y_val, oof_xgb[val_idx])
        print(f'    fold {fold:02d} | LGBM={r2_l:.4f}  XGB={r2_x:.4f}')

    print(f'  OOF  LGBM={r2_score(y_train, oof_lgbm):.4f}  '
          f'XGB={r2_score(y_train, oof_xgb):.4f}')

    return (oof_lgbm, oof_xgb), (test_lgbm, test_xgb)


print('Training function defined.')

In [ ]:
print('=' * 60)
print('  Tabular — Tg  (LGBM + XGB)')
print('=' * 60)

tg_oof_parts_all  = []
tg_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(X_tg, y_tg, X_tg_test, 'tg', seed=seed)
    tg_oof_parts_all.append(oof_parts)
    tg_test_parts_all.append(test_parts)

oof_l_tg  = np.mean([p[0] for p in tg_oof_parts_all],  axis=0)
oof_x_tg  = np.mean([p[1] for p in tg_oof_parts_all],  axis=0)
test_l_tg = np.mean([p[0] for p in tg_test_parts_all], axis=0)
test_x_tg = np.mean([p[1] for p in tg_test_parts_all], axis=0)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_tg, oof_l_tg):.4f}  '
      f'XGB={r2_score(y_tg, oof_x_tg):.4f}')


In [ ]:
print('=' * 60)
print('  Tabular — Egc  (LGBM + XGB)')
print('=' * 60)

egc_oof_parts_all  = []
egc_test_parts_all = []

for seed in SEEDS:
    print(f'\n  seed={seed}')
    oof_parts, test_parts = train_ensemble(X_egc, y_egc, X_egc_test, 'egc', seed=seed)
    egc_oof_parts_all.append(oof_parts)
    egc_test_parts_all.append(test_parts)

oof_l_egc  = np.mean([p[0] for p in egc_oof_parts_all],  axis=0)
oof_x_egc  = np.mean([p[1] for p in egc_oof_parts_all],  axis=0)
test_l_egc = np.mean([p[0] for p in egc_test_parts_all], axis=0)
test_x_egc = np.mean([p[1] for p in egc_test_parts_all], axis=0)

print(f'\nSeed-averaged OOF R²  LGBM={r2_score(y_egc, oof_l_egc):.4f}  '
      f'XGB={r2_score(y_egc, oof_x_egc):.4f}')


## Optimise 3-Way Blend Weights on OOF

Now blending **LGBM + XGB + GNN**. The constrained optimisation finds the weight vector that maximises OOF R² per target. GNN diversity (error correlation 0.765 with XGB) should earn it a real, non-zero weight — unlike ExtraTrees (0.82 correlation → 0 weight) and CatBoost (similar GBDT family → 0 weight).

In [ ]:
from scipy.optimize import minimize

def find_best_weights(parts, y_true):
    stack = np.column_stack(parts)
    def neg_r2(w):
        return -r2_score(y_true, stack @ w)
    n = len(parts)
    res = minimize(
        neg_r2, x0=[1/n]*n, method='SLSQP',
        bounds=[(0, 1)] * n,
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
    )
    return res.x

w_tg  = find_best_weights((oof_l_tg, oof_x_tg, oof_g_tg),   y_tg)
w_egc = find_best_weights((oof_l_egc, oof_x_egc, oof_g_egc), y_egc)

print(f'Optimal Tg  weights — LGBM: {w_tg[0]:.3f}  XGB: {w_tg[1]:.3f}  GNN: {w_tg[2]:.3f}')
print(f'Optimal Egc weights — LGBM: {w_egc[0]:.3f}  XGB: {w_egc[1]:.3f}  GNN: {w_egc[2]:.3f}')

oof_tg_opt  = w_tg[0]*oof_l_tg   + w_tg[1]*oof_x_tg   + w_tg[2]*oof_g_tg
oof_egc_opt = w_egc[0]*oof_l_egc  + w_egc[1]*oof_x_egc  + w_egc[2]*oof_g_egc

r2_tg  = r2_score(y_tg,  oof_tg_opt)
r2_egc = r2_score(y_egc, oof_egc_opt)

print(f'\nOOF R² Tg  : {r2_tg:.4f}   (v6, no GNN: 0.9082)')
print(f'OOF R² Egc : {r2_egc:.4f}   (v6, no GNN: 0.9151)')
print(f'Mean OOF R²: {(r2_tg + r2_egc)/2:.4f}   (v6, no GNN: 0.9117)')

pred_tg  = w_tg[0]*test_l_tg   + w_tg[1]*test_x_tg   + w_tg[2]*test_g_tg
pred_egc = w_egc[0]*test_l_egc  + w_egc[1]*test_x_egc  + w_egc[2]*test_g_egc

print('\nTest predictions updated with optimised 3-way weights.')


In [ ]:
sub_tg          = test_tg[['id']].copy()
sub_tg['target'] = pred_tg

sub_egc          = test_egc[['id']].copy()
sub_egc['target'] = pred_egc

submission = (
    pd.concat([sub_tg, sub_egc], axis=0)
    .sort_values('id')
    .reset_index(drop=True)
)

assert submission.shape[0] == len(test), 'Row count mismatch!'
assert submission['target'].isna().sum() == 0, 'NaN in predictions!'

submission.to_csv('/kaggle/working/submission.csv', index=False)
print('Submission shape:', submission.shape)
print(submission.head(10))
print('\nsubmission.csv saved.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_tg, oof_tg_opt, alpha=0.25, s=8)
lo, hi = min(y_tg.min(), oof_tg_opt.min()), max(y_tg.max(), oof_tg_opt.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[0].set_xlabel('True Tg (°C)', fontsize=12)
axes[0].set_ylabel('Pred Tg (°C)', fontsize=12)
axes[0].set_title(f'Tg OOF  R² = {r2_tg:.4f}', fontsize=13)
axes[0].grid(alpha=0.3)

axes[1].scatter(y_egc, oof_egc_opt, alpha=0.25, s=8, color='darkorange')
lo, hi = min(y_egc.min(), oof_egc_opt.min()), max(y_egc.max(), oof_egc_opt.max())
axes[1].plot([lo, hi], [lo, hi], 'r--', lw=1.5)
axes[1].set_xlabel('True Egc (eV)', fontsize=12)
axes[1].set_ylabel('Pred Egc (eV)', fontsize=12)
axes[1].set_title(f'Egc OOF  R² = {r2_egc:.4f}', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f'v8 OOF Predicted vs Actual  |  Mean R² = {(r2_tg+r2_egc)/2:.4f}   (v6 (no GNN): 0.9117)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/kaggle/working/oof_scatter_v8.png', dpi=120, bbox_inches='tight')
plt.show()
print('Plot saved.')